# 🌍 Climate Change Modeling — NLP & Sentiment Analysis
**Organization:** Unified Mentor | **Domain:** Data Science | **Level:** Advanced

### Project Pipeline
1. Install & Import Libraries
2. Load & Explore Dataset
3. Data Preprocessing
4. Exploratory Data Analysis (EDA)
5. Text Preprocessing & NLP
6. Sentiment Analysis
7. Topic Modeling
8. Feature Engineering
9. ML Model Training & Evaluation
10. Future Projections & Insights

---
## 📦 Step 1: Install & Import Libraries

In [ ]:
# ── Install required libraries (Colab-safe) ──────────────────────────────────
!pip install -q textblob vaderSentiment wordcloud gensim pyLDAvis

In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import string
import warnings
warnings.filterwarnings('ignore')

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from wordcloud import WordCloud
%matplotlib inline
sns.set_style('darkgrid')
PALETTE = ['#2196F3','#4CAF50','#FF5722','#9C27B0','#FF9800']

# ── NLP ───────────────────────────────────────────────────────────────────────
import nltk
nltk.download('punkt',      quiet=True)
nltk.download('stopwords',  quiet=True)
nltk.download('wordnet',    quiet=True)
nltk.download('punkt_tab',  quiet=True)
from nltk.corpus   import stopwords
from nltk.stem     import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from textblob      import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from collections   import Counter

# ── Topic Modeling ────────────────────────────────────────────────────────────
from gensim import corpora, models
from gensim.models.ldamodel import LdaModel

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection  import train_test_split, cross_val_score, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing    import LabelEncoder, StandardScaler
from sklearn.ensemble         import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model     import LogisticRegression
from sklearn.naive_bayes      import MultinomialNB
from sklearn.svm              import LinearSVC
from sklearn.metrics          import (classification_report, confusion_matrix,
                                       accuracy_score, f1_score)
from sklearn.pipeline         import Pipeline

print('✅ All libraries imported successfully!')

---
## 📂 Step 2: Load & Explore Dataset

In [ ]:
# ── Upload the CSV from your local machine ────────────────────────────────────
from google.colab import files
uploaded = files.upload()          # select  climate_nasa.csv  when prompted

import io
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Dataset loaded  →  {df.shape[0]} rows × {df.shape[1]} columns')

In [ ]:
# ── First look ────────────────────────────────────────────────────────────────
print('=== Shape ===')
print(df.shape)

print('\n=== Columns ===')
print(df.columns.tolist())

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== First 5 rows ===')
df.head()

In [ ]:
# ── Statistical summary ───────────────────────────────────────────────────────
print('=== Descriptive Statistics ===')
df.describe(include='all')

In [ ]:
# ── Missing values ────────────────────────────────────────────────────────────
print('=== Missing Values ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# ── Duplicate check ───────────────────────────────────────────────────────────
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Duplicate texts: {df["text"].duplicated().sum()}')

---
## 🔧 Step 3: Data Preprocessing

In [ ]:
# ── 3.1  Drop rows with missing text (18 rows) ────────────────────────────────
df = df.dropna(subset=['text']).reset_index(drop=True)
print(f'Rows after dropping missing text: {len(df)}')

# ── 3.2  Fill missing commentsCount with 0 ───────────────────────────────────
df['commentsCount'] = df['commentsCount'].fillna(0).astype(int)

# ── 3.3  Parse datetime ───────────────────────────────────────────────────────
df['date'] = pd.to_datetime(df['date'], utc=True)
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day
df['hour']  = df['date'].dt.hour
df['month_name'] = df['date'].dt.strftime('%b')
df['day_of_week'] = df['date'].dt.day_name()

print('\n=== Date range ===')
print(f"From: {df['date'].min().date()}  →  To: {df['date'].max().date()}")
print(f'Years covered: {sorted(df.year.unique())}')

In [ ]:
# ── 3.4  Text length features (before cleaning) ───────────────────────────────
df['text_length']  = df['text'].apply(len)
df['word_count']   = df['text'].apply(lambda x: len(str(x).split()))
df['char_count']   = df['text'].apply(lambda x: len(str(x).replace(' ','')))
df['sentence_count'] = df['text'].apply(lambda x: len(re.split(r'[.!?]+', str(x))))

print('Text length stats:')
df[['text_length','word_count','char_count','sentence_count']].describe().round(2)

---
## 📊 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1  Comments per Year ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Temporal Distribution of Comments', fontsize=16, fontweight='bold')

# Year
year_counts = df['year'].value_counts().sort_index()
axes[0].bar(year_counts.index.astype(str), year_counts.values, color=PALETTE[:len(year_counts)])
axes[0].set_title('Comments per Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')
for i, v in enumerate(year_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')

# Month
month_order = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_counts = df['month_name'].value_counts().reindex(month_order).fillna(0)
axes[1].bar(month_counts.index, month_counts.values, color=PALETTE[1])
axes[1].set_title('Comments per Month')
axes[1].set_xlabel('Month')
axes[1].tick_params(axis='x', rotation=45)

# Hour
hour_counts = df['hour'].value_counts().sort_index()
axes[2].plot(hour_counts.index, hour_counts.values, color=PALETTE[2], marker='o', linewidth=2)
axes[2].fill_between(hour_counts.index, hour_counts.values, alpha=0.3, color=PALETTE[2])
axes[2].set_title('Comments by Hour of Day')
axes[2].set_xlabel('Hour (UTC)')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2  Engagement Analysis (Likes & Comments) ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Engagement Analysis', fontsize=16, fontweight='bold')

# Likes distribution
sns.histplot(df['likesCount'], bins=30, kde=True, ax=axes[0], color=PALETTE[0])
axes[0].set_title('Distribution of Likes per Comment')
axes[0].set_xlabel('Likes Count')
axes[0].set_ylabel('Frequency')

# Avg likes per year
avg_likes = df.groupby('year')['likesCount'].mean()
axes[1].bar(avg_likes.index.astype(str), avg_likes.values, color=PALETTE[3])
axes[1].set_title('Average Likes per Comment by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Avg Likes')
for i, v in enumerate(avg_likes.values):
    axes[1].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 4.3  Text Length Distribution ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Text Characteristics', fontsize=16, fontweight='bold')

sns.histplot(df['word_count'], bins=40, kde=True, ax=axes[0], color=PALETTE[1])
axes[0].set_title('Word Count Distribution')
axes[0].set_xlabel('Words per Comment')
axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--',
                label=f'Mean: {df["word_count"].mean():.1f}')
axes[0].legend()

sns.boxplot(data=df, x='year', y='word_count', ax=axes[1], palette=PALETTE[:4])
axes[1].set_title('Word Count per Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.show()
print(f'Average words per comment: {df["word_count"].mean():.1f}')
print(f'Skewness of word count: {df["word_count"].skew():.2f}')

---
## 🔤 Step 5: Text Preprocessing & NLP

In [ ]:
# ── 5.1  Text Cleaning Function ───────────────────────────────────────────────
STOP_WORDS = set(stopwords.words('english'))
LEMMATIZER = WordNetLemmatizer()

# Add climate-context stop words (very common, low-value words in this domain)
EXTRA_STOPS = {'climate','change','nasa','global','earth','world','would',
               'people','think','know','one','also','get','like','even',
               'much','many','well','said','say','go','us','make','way'}
STOP_WORDS.update(EXTRA_STOPS)

def clean_text(text):
    """Full cleaning pipeline: lowercase → URLs → mentions → punctuation
       → numbers → tokenize → stopwords → lemmatize."""
    text = str(text).lower()                           # lowercase
    text = re.sub(r'http\S+|www\.\S+', '', text)      # remove URLs
    text = re.sub(r'@\w+', '', text)                   # remove @mentions
    text = re.sub(r'#\w+', '', text)                   # remove hashtags
    text = re.sub(r'[^a-z\s]', ' ', text)             # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()           # collapse spaces
    tokens = word_tokenize(text)
    tokens = [LEMMATIZER.lemmatize(t) for t in tokens
              if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['text'].apply(clean_text)
print('✅ Text cleaning complete')
print('\nOriginal:  ', df['text'].iloc[0][:120])
print('\nCleaned:   ', df['clean_text'].iloc[0][:120])

In [ ]:
# ── 5.2  WordCloud ────────────────────────────────────────────────────────────
all_words = ' '.join(df['clean_text'].dropna())

wc = WordCloud(
    width=1000, height=500,
    background_color='white',
    colormap='viridis',
    max_words=150,
    collocations=False
).generate(all_words)

plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Words in NASA Climate Change Comments',
          fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3  Top 20 most frequent words ──────────────────────────────────────────
all_tokens = all_words.split()
word_freq  = Counter(all_tokens).most_common(20)
words_df   = pd.DataFrame(word_freq, columns=['Word', 'Frequency'])

plt.figure(figsize=(12, 6))
sns.barplot(data=words_df, x='Frequency', y='Word', palette='Blues_r')
plt.title('Top 20 Most Frequent Words', fontsize=15, fontweight='bold')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()

---
## 😊 Step 6: Sentiment Analysis

In [ ]:
# ── 6.1  VADER Sentiment (best for social media text) ────────────────────────
vader = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    scores = vader.polarity_scores(str(text))
    return pd.Series({
        'vader_neg'      : scores['neg'],
        'vader_neu'      : scores['neu'],
        'vader_pos'      : scores['pos'],
        'vader_compound' : scores['compound']
    })

vader_scores = df['text'].apply(get_vader_scores)
df = pd.concat([df, vader_scores], axis=1)

# ── Classify sentiment from compound score ────────────────────────────────────
def classify_vader(score):
    if score >= 0.05:  return 'Positive'
    elif score <= -0.05: return 'Negative'
    else: return 'Neutral'

df['vader_sentiment'] = df['vader_compound'].apply(classify_vader)
print('✅ VADER Sentiment Analysis complete')
print('\nSentiment Distribution:')
print(df['vader_sentiment'].value_counts())

In [ ]:
# ── 6.2  TextBlob Sentiment ───────────────────────────────────────────────────
def get_textblob_sentiment(text):
    analysis = TextBlob(str(text))
    pol  = analysis.sentiment.polarity
    subj = analysis.sentiment.subjectivity
    if pol > 0.05:   label = 'Positive'
    elif pol < -0.05: label = 'Negative'
    else:             label = 'Neutral'
    return pd.Series({'tb_polarity': pol, 'tb_subjectivity': subj,
                      'tb_sentiment': label})

tb = df['text'].apply(get_textblob_sentiment)
df = pd.concat([df, tb], axis=1)
print('✅ TextBlob Sentiment Analysis complete')
print('\nTextBlob Sentiment Distribution:')
print(df['tb_sentiment'].value_counts())

In [ ]:
# ── 6.3  Sentiment Visualizations ────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Sentiment Analysis Dashboard', fontsize=17, fontweight='bold')

# VADER sentiment pie
vader_counts = df['vader_sentiment'].value_counts()
axes[0,0].pie(vader_counts, labels=vader_counts.index,
              autopct='%1.1f%%', colors=['#4CAF50','#9E9E9E','#F44336'],
              startangle=90, textprops={'fontsize':12})
axes[0,0].set_title('VADER Sentiment Distribution', fontweight='bold')

# TextBlob sentiment pie
tb_counts = df['tb_sentiment'].value_counts()
axes[0,1].pie(tb_counts, labels=tb_counts.index,
              autopct='%1.1f%%', colors=['#2196F3','#9E9E9E','#FF5722'],
              startangle=90, textprops={'fontsize':12})
axes[0,1].set_title('TextBlob Sentiment Distribution', fontweight='bold')

# VADER compound score histogram
sns.histplot(df['vader_compound'], bins=30, kde=True, ax=axes[0,2], color=PALETTE[3])
axes[0,2].axvline(0, color='black', linestyle='--', alpha=0.6)
axes[0,2].set_title('VADER Compound Score Distribution', fontweight='bold')
axes[0,2].set_xlabel('Compound Score')

# Sentiment trend over years
sentiment_year = df.groupby(['year','vader_sentiment']).size().unstack(fill_value=0)
sentiment_year.plot(kind='bar', ax=axes[1,0], color=['#F44336','#9E9E9E','#4CAF50'],
                   edgecolor='white')
axes[1,0].set_title('Sentiment Trend by Year', fontweight='bold')
axes[1,0].set_xlabel('Year')
axes[1,0].tick_params(axis='x', rotation=0)
axes[1,0].legend(title='Sentiment')

# Subjectivity distribution
sns.histplot(df['tb_subjectivity'], bins=30, kde=True, ax=axes[1,1], color=PALETTE[1])
axes[1,1].set_title('Subjectivity Score Distribution\n(0=Objective, 1=Subjective)',
                    fontweight='bold')
axes[1,1].set_xlabel('Subjectivity')

# Avg likes by sentiment
avg_likes_sent = df.groupby('vader_sentiment')['likesCount'].mean().sort_values()
axes[1,2].bar(avg_likes_sent.index, avg_likes_sent.values,
              color=['#F44336','#9E9E9E','#4CAF50'])
axes[1,2].set_title('Average Likes by Sentiment', fontweight='bold')
axes[1,2].set_xlabel('Sentiment')
axes[1,2].set_ylabel('Avg Likes')
for i, v in enumerate(avg_likes_sent.values):
    axes[1,2].text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 6.4  Sample comments per sentiment ───────────────────────────────────────
for sent in ['Positive', 'Negative', 'Neutral']:
    sample = df[df['vader_sentiment'] == sent]['text'].iloc[0]
    score  = df[df['vader_sentiment'] == sent]['vader_compound'].iloc[0]
    print(f'\n[{sent}] (score={score:.3f})')
    print(f'  {sample[:200]}')

---
## 🗂️ Step 7: Topic Modeling (LDA)

In [ ]:
# ── 7.1  Build LDA Model ──────────────────────────────────────────────────────
# Tokenize cleaned text
tokenized = df['clean_text'].dropna().apply(lambda x: x.split()).tolist()

# Build dictionary and corpus
dictionary = corpora.Dictionary(tokenized)
dictionary.filter_extremes(no_below=5, no_above=0.85)  # remove very rare/common
corpus = [dictionary.doc2bow(doc) for doc in tokenized]

# Train LDA
NUM_TOPICS = 5
lda_model = LdaModel(
    corpus       = corpus,
    id2word      = dictionary,
    num_topics   = NUM_TOPICS,
    random_state = 42,
    passes       = 15,
    alpha        = 'auto',
    per_word_topics = True
)

print(f'✅ LDA model trained with {NUM_TOPICS} topics')
print(f'   Perplexity : {lda_model.log_perplexity(corpus):.2f}')

In [ ]:
# ── 7.2  Display Topics ───────────────────────────────────────────────────────
TOPIC_LABELS = [
    'Scientific Evidence',
    'Political & Policy Debate',
    'Weather & Observations',
    'Renewable Energy & Solutions',
    'Public Concern & Emotions'
]

print('\n🗂️  Discovered Topics:\n')
print(f'{"Topic":<30} {"Top Keywords"}')
print('-' * 80)
for idx, topic in lda_model.print_topics(num_words=8):
    label = TOPIC_LABELS[idx] if idx < len(TOPIC_LABELS) else f'Topic {idx}'
    keywords = ' | '.join([w.split('*')[1].replace('"','').strip()
                           for w in topic.split(' + ')])
    print(f'Topic {idx} ({label:<28}): {keywords}')

In [ ]:
# ── 7.3  Assign dominant topic to each comment ────────────────────────────────
def get_dominant_topic(bow):
    topics = lda_model.get_document_topics(bow)
    if not topics:
        return -1
    return max(topics, key=lambda x: x[1])[0]

valid_idx     = df['clean_text'].dropna().index
corpus_all    = [dictionary.doc2bow(t) for t in
                 df.loc[valid_idx, 'clean_text'].apply(lambda x: x.split())]
df.loc[valid_idx, 'dominant_topic'] = [get_dominant_topic(c) for c in corpus_all]
df['topic_label'] = df['dominant_topic'].map(
    {i: TOPIC_LABELS[i] for i in range(len(TOPIC_LABELS))})

# Visualize topic distribution
plt.figure(figsize=(10, 5))
topic_counts = df['topic_label'].value_counts()
sns.barplot(x=topic_counts.values, y=topic_counts.index, palette='viridis')
plt.title('Document Distribution Across Topics', fontsize=14, fontweight='bold')
plt.xlabel('Number of Comments')
for i, v in enumerate(topic_counts.values):
    plt.text(v + 1, i, str(v), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## ⚙️ Step 8: Feature Engineering for ML

In [ ]:
# ── 8.1  Create ML-ready features ────────────────────────────────────────────
# Encode target label
le = LabelEncoder()
df['sentiment_label'] = le.fit_transform(df['vader_sentiment'])  # 0=Neg, 1=Neu, 2=Pos

# Numeric features
NUM_FEATURES = ['likesCount','commentsCount','word_count','char_count',
                'sentence_count','tb_polarity','tb_subjectivity',
                'vader_neg','vader_neu','vader_pos','hour','month']

print('✅ Features prepared')
print(f'   Label mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}')
print(f'   Class distribution:\n{df["vader_sentiment"].value_counts()}')

In [ ]:
# ── 8.2  TF-IDF Vectorization ─────────────────────────────────────────────────
# Use clean_text for TF-IDF; drop rows where clean_text is empty
ml_df = df[df['clean_text'].str.len() > 5].copy().reset_index(drop=True)

X_text = ml_df['clean_text']
y      = ml_df['sentiment_label']

tfidf = TfidfVectorizer(
    max_features = 3000,
    ngram_range  = (1, 2),   # unigrams + bigrams
    min_df       = 2,
    max_df       = 0.90,
    sublinear_tf = True
)

X_tfidf = tfidf.fit_transform(X_text)
print(f'TF-IDF matrix shape: {X_tfidf.shape}')
print(f'Target classes: {le.classes_}  → labels: {le.transform(le.classes_)}')

In [ ]:
# ── 8.3  Train-Test Split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y)

print(f'Train size : {X_train.shape[0]}')
print(f'Test  size : {X_test.shape[0]}')

---
## 🤖 Step 9: ML Model Training & Evaluation

In [ ]:
# ── 9.1  Train Multiple Models ───────────────────────────────────────────────
MODELS = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Naive Bayes'         : MultinomialNB(alpha=0.1),
    'Linear SVM'          : LinearSVC(max_iter=2000, C=1.0, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42,
                                                   n_jobs=-1),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results = []
trained_models = {}

print('Training models...\n')
for name, model in MODELS.items():
    model.fit(X_train, y_train)
    y_pred    = model.predict(X_test)
    acc       = accuracy_score(y_test, y_pred)
    f1_macro  = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    # 5-fold cross-validation
    cv_scores = cross_val_score(model, X_tfidf, y, cv=5,
                                scoring='accuracy', n_jobs=-1)

    results.append({
        'Model'       : name,
        'Accuracy'    : round(acc, 4),
        'F1-Macro'    : round(f1_macro, 4),
        'F1-Weighted' : round(f1_weighted, 4),
        'CV-Mean'     : round(cv_scores.mean(), 4),
        'CV-Std'      : round(cv_scores.std(), 4)
    })
    trained_models[name] = (model, y_pred)
    print(f'  ✅ {name:<25} Acc={acc:.4f}  F1={f1_macro:.4f}  CV={cv_scores.mean():.4f}±{cv_scores.std():.4f}')

results_df = pd.DataFrame(results).sort_values('CV-Mean', ascending=False)
print('\n=== Model Comparison Table ===')
results_df

In [ ]:
# ── 9.2  Model Performance Bar Chart ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(results_df))
width = 0.25

bars1 = ax.bar(x - width, results_df['Accuracy'],    width, label='Accuracy',    color=PALETTE[0])
bars2 = ax.bar(x,         results_df['F1-Macro'],    width, label='F1-Macro',    color=PALETTE[1])
bars3 = ax.bar(x + width, results_df['CV-Mean'],     width, label='CV Accuracy', color=PALETTE[2])

ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Model Comparison — Accuracy, F1-Macro & CV Score',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right')
ax.set_ylim(0, 1.0)
ax.legend()
ax.axhline(0.8, color='red', linestyle='--', alpha=0.5, label='80% threshold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3  Best model — Confusion Matrix & Classification Report ────────────────
best_name = results_df.iloc[0]['Model']
best_model, best_preds = trained_models[best_name]

print(f'🏆 Best Model: {best_name}')
print('\n=== Classification Report ===')
print(classification_report(y_test, best_preds,
                             target_names=le.classes_))

# Confusion matrix
cm = confusion_matrix(y_test, best_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.4  Feature Importance (Top 20 TF-IDF words) ─────────────────────────────
# Works for Logistic Regression and Linear SVM
lr_model = trained_models['Logistic Regression'][0]
feature_names = np.array(tfidf.get_feature_names_out())

# Get top features per class
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Top 15 TF-IDF Features per Sentiment Class\n(Logistic Regression Coefficients)',
             fontsize=14, fontweight='bold')

colors_map = {'Negative': '#F44336', 'Neutral': '#9E9E9E', 'Positive': '#4CAF50'}

for ax, (cls_idx, cls_name) in zip(axes, enumerate(le.classes_)):
    coef = lr_model.coef_[cls_idx]
    top_idx = np.argsort(coef)[-15:]
    top_words = feature_names[top_idx]
    top_coefs = coef[top_idx]
    ax.barh(top_words, top_coefs, color=colors_map[cls_name])
    ax.set_title(f'{cls_name} Sentiment', fontweight='bold')
    ax.set_xlabel('Coefficient')

plt.tight_layout()
plt.show()

---
## 📈 Step 10: Trend Analysis & Insights

In [ ]:
# ── 10.1  Sentiment Trend over Time ──────────────────────────────────────────
# Monthly sentiment average
df['year_month'] = df['date'].dt.to_period('M').astype(str)
monthly_sentiment = df.groupby('year_month')['vader_compound'].mean().reset_index()
monthly_sentiment.columns = ['year_month', 'avg_sentiment']

plt.figure(figsize=(14, 5))
plt.plot(range(len(monthly_sentiment)), monthly_sentiment['avg_sentiment'],
         color=PALETTE[0], linewidth=2, marker='o', markersize=4)
plt.fill_between(range(len(monthly_sentiment)), monthly_sentiment['avg_sentiment'],
                 alpha=0.15, color=PALETTE[0])
plt.axhline(0, color='red', linestyle='--', alpha=0.5, label='Neutral baseline')

# X-axis: show only every 3rd label
ticks = range(0, len(monthly_sentiment), 3)
plt.xticks(ticks, [monthly_sentiment['year_month'].iloc[i] for i in ticks], rotation=45)
plt.title('Monthly Average Sentiment Trend (2020–2023)', fontsize=14, fontweight='bold')
plt.ylabel('Average VADER Compound Score')
plt.xlabel('Month')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.2  Engagement Correlation Heatmap ─────────────────────────────────────
corr_cols = ['likesCount','commentsCount','word_count','vader_compound',
             'tb_polarity','tb_subjectivity','sentiment_label']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5)
plt.title('Correlation Heatmap — Engagement & Sentiment Features',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.3  Final Summary Statistics ───────────────────────────────────────────
pos_pct  = (df['vader_sentiment']=='Positive').mean()*100
neg_pct  = (df['vader_sentiment']=='Negative').mean()*100
neu_pct  = (df['vader_sentiment']=='Neutral').mean()*100
avg_sent = df['vader_compound'].mean()
avg_subj = df['tb_subjectivity'].mean()
best_acc = results_df.iloc[0]['CV-Mean']

print('=' * 60)
print('  📊 PROJECT SUMMARY — Climate Change NLP Analysis')
print('=' * 60)
print(f'  Total Comments Analyzed : {len(df)}')
print(f'  Date Range              : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Positive Comments       : {pos_pct:.1f}%')
print(f'  Negative Comments       : {neg_pct:.1f}%')
print(f'  Neutral  Comments       : {neu_pct:.1f}%')
print(f'  Avg Sentiment Score     : {avg_sent:.3f} ({"Positive" if avg_sent>0 else "Negative"})')
print(f'  Avg Subjectivity        : {avg_subj:.3f} ({"Subjective" if avg_subj>0.5 else "Objective"})')
print(f'  Best ML Model           : {best_name}')
print(f'  Best CV Accuracy        : {best_acc:.4f} ({best_acc*100:.1f}%)')
print(f'  Topics Discovered       : {NUM_TOPICS}')
print('=' * 60)

In [ ]:
# ── 10.4  Predict sentiment on new custom text ────────────────────────────────
def predict_sentiment(text, model=best_model, vectorizer=tfidf):
    """Predict sentiment label and scores for any input text."""
    cleaned    = clean_text(text)
    tfidf_vec  = vectorizer.transform([cleaned])
    pred_label = le.inverse_transform(model.predict(tfidf_vec))[0]
    vader_score = vader.polarity_scores(text)['compound']
    tb_score    = TextBlob(text).sentiment.polarity
    print(f'Input     : {text[:100]}')
    print(f'Prediction: {pred_label}')
    print(f'VADER     : {vader_score:.3f}')
    print(f'TextBlob  : {tb_score:.3f}')
    print()

# Test examples
predict_sentiment("Climate change is an urgent crisis and we must act now!")
predict_sentiment("I don't believe the climate data from NASA is accurate at all.")
predict_sentiment("The temperature data for this region shows mixed signals.")

---
## 💾 Save Processed Dataset

In [ ]:
# Save the enriched dataset with sentiment & topic labels
output_cols = ['date','year','month','hour','likesCount','commentsCount',
               'text','clean_text','word_count','vader_compound',
               'vader_sentiment','tb_polarity','tb_subjectivity',
               'tb_sentiment','topic_label']

df[output_cols].to_csv('climate_nasa_processed.csv', index=False)

# Download it
from google.colab import files
files.download('climate_nasa_processed.csv')
print('✅ Processed dataset downloaded!')